# Dev evaluation: ICD-condition cosine similarity

For each held-out test note, the model extracts conditions and matches them to columns from the training-only CSV. Python ranks ICD codes by cosine similarity between the note's binary condition vector and each CSV row. Unmatched conditions are ignored; no CSV columns are added. Up to three positive-score codes are written for `score.py`. Ties are ordered by ICD code; if there is no overlap, the prediction is empty.

**Important:** each test gold code is absent from the training split, so this closed-set method cannot predict any of them and its score will be 0. First build `out/icd_conditions_train.csv` using the training notebook. Running the prediction cell sends note text to the configured model provider, makes up to two API calls per note, and may incur charges. It overwrites the holdout output files in `out/`. Saved outputs from earlier runs may show the old full-data results; rerun the cells for this split.

In [1]:
import csv
import html
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
import condition_utils
from score import norm

root = Path.cwd()
if not (root / 'llm.py').is_file() or not (root / 'score.py').is_file():
    raise FileNotFoundError('Start the notebook with the project root as its working directory.')
provider = 'openai'
model = None  # None uses the provider's MODEL environment variable or llm.py default
api_python = '/opt/homebrew/bin/python3'  # Python that can verify your network's TLS certificate
gold_path = root / 'data' / 'dev_notes_test.jsonl'
train_path = root / 'data' / 'dev_notes_train.jsonl'
mms_path = root / 'data' / 'mms-2026-01.json'
matrix_path = root / 'out' / 'icd_conditions_train.csv'
predictions_path = root / 'out' / 'dev_predictions_test02_holdout.jsonl'
similarity_path = root / 'out' / 'dev_similarity_test02_holdout.jsonl'
per_note_path = root / 'out' / 'dev_per_note_test02_holdout.jsonl'

## Load and validate the inputs

The CSV is read only. The extraction and matching prompts are shared with the database-building notebook.

In [2]:
with gold_path.open(encoding='utf-8') as file:
    notes = [json.loads(line) for line in file if line.strip()]
with train_path.open(encoding='utf-8') as file:
    training_notes = [json.loads(line) for line in file if line.strip()]
for note in notes + training_notes:
    if not isinstance(note, dict) or not isinstance(note.get('id'), str) or not isinstance(note.get('text'), str) or not isinstance(note.get('gold'), list) or not note['gold'] or any(not norm(code) for code in note['gold']):
        raise ValueError('Each note must have an id, text, and non-empty gold codes.')
if {note['id'] for note in notes} & {note['id'] for note in training_notes}:
    raise ValueError('Training and test notes must have disjoint IDs.')
with matrix_path.open(encoding='utf-8', newline='') as file:
    header = next(csv.reader(file))
if not header or header[0] != 'icd_code' or len(header) < 2:
    raise ValueError('The CSV must have an icd_code column followed by condition columns.')
if any(not column.strip() for column in header) or len({column.casefold() for column in header}) != len(header):
    raise ValueError('CSV column names must be non-empty and unique, ignoring case.')
df = pd.read_csv(matrix_path, dtype=str, keep_default_na=False)
if df.empty or df['icd_code'].duplicated().any() or any(not code.strip() for code in df['icd_code']):
    raise ValueError('The CSV must contain unique, non-empty ICD codes.')
training_codes = {norm(code) for note in training_notes for code in note['gold']}
matrix_codes = {norm(code) for code in df['icd_code']}
if matrix_codes != training_codes:
    raise ValueError('The CSV codes do not match the training gold codes. Rebuild the training CSV before evaluation.')
covered = sum(any(norm(code) in matrix_codes for code in note['gold']) for note in notes)
print(f'Test notes with at least one gold code in the training candidates: {covered}/{len(notes)}')
if covered == 0:
    print('WARNING: No test gold codes are candidates; the closed-set score must be 0.')
columns = list(df.columns[1:])
if not df[columns].isin(['0', '1']).all().all():
    raise ValueError('Every condition value in the CSV must be 0 or 1.')
code_features = [
    (row[0], {column for column, value in zip(columns, row[1:]) if value == '1'})
    for row in df.itertuples(index=False, name=None)
]
extract_prompt = (root / 'prompts' / 'extract_conditions.txt').read_text(encoding='utf-8').strip()
match_prompt = (root / 'prompts' / 'match_conditions.txt').read_text(encoding='utf-8').strip()
if not extract_prompt or not match_prompt:
    raise ValueError('Extraction and matching prompts must not be empty.')
match_prompt += '\nFor this evaluation, null means ignore the condition; do not propose or create new columns.'
print(f'Loaded {len(notes)} notes, {len(df)} ICD codes and {len(columns)} condition columns.')

Test notes with at least one gold code in the training candidates: 0/10
Loaded 10 notes, 51 ICD codes and 193 condition columns.


## Match conditions and rank ICD codes

Cosine similarity is `overlap / sqrt(note_condition_count * ICD_condition_count)`. Python computes the scores and handles missing matches; the model does not assign scores or 0/1 values.

In [3]:
def call_model(prompt, input_text):
    return condition_utils.call_model(prompt, input_text, root=root, provider=provider, model=model, api_python=api_python)

## Generate predictions

Each result is flushed to disk as it completes. If a call fails, the partial prediction file can still be scored, but rerunning this cell starts over.

In [4]:
predictions_path.parent.mkdir(parents=True, exist_ok=True)
with predictions_path.open('w', encoding='utf-8') as predictions, similarity_path.open('w', encoding='utf-8') as details:
    for index, note in enumerate(notes, 1):
        print(f"[{index}/{len(notes)}] {note['id']}: extracting conditions", flush=True)
        conditions = condition_utils.extract_conditions(note['text'], extract_prompt, call_model)
        if conditions:
            print(f"[{index}/{len(notes)}] {note['id']}: matching existing columns", flush=True)
            mappings = condition_utils.match_conditions(conditions, columns, match_prompt, call_model)
        else:
            mappings = {}
        matched, unmatched = condition_utils.existing_matches(conditions, mappings, columns)
        ranked = condition_utils.rank_codes(matched, code_features)
        predictions.write(json.dumps({'id': note['id'], 'codes': [code for code, score in ranked]}, ensure_ascii=False) + '\n')
        predictions.flush()
        details.write(json.dumps({'id': note['id'], 'matched_conditions': sorted(matched), 'unmatched_conditions': unmatched, 'ranked': [{'code': code, 'similarity': score} for code, score in ranked]}, ensure_ascii=False) + '\n')
        details.flush()
print(f'Predictions: {predictions_path}\nSimilarity details: {similarity_path}')

[1/10] dev-051: extracting conditions
[1/10] dev-051: matching existing columns
[2/10] dev-052: extracting conditions
[2/10] dev-052: matching existing columns
[3/10] dev-053: extracting conditions
[3/10] dev-053: matching existing columns
[4/10] dev-054: extracting conditions
[4/10] dev-054: matching existing columns
[5/10] dev-055: extracting conditions
[5/10] dev-055: matching existing columns
[6/10] dev-056: extracting conditions
[6/10] dev-056: matching existing columns
[7/10] dev-057: extracting conditions
[7/10] dev-057: matching existing columns
[8/10] dev-058: extracting conditions
[8/10] dev-058: matching existing columns
[9/10] dev-059: extracting conditions
[9/10] dev-059: matching existing columns
[10/10] dev-060: extracting conditions
[10/10] dev-060: matching existing columns
Predictions: /Users/shivangi.s.1/Desktop/sugen_test/out/dev_predictions_test02_holdout.jsonl
Similarity details: /Users/shivangi.s.1/Desktop/sugen_test/out/dev_similarity_test02_holdout.jsonl


## Evaluate using the official scorer

This reports the held-out top-1, top-3, and weighted score. Missing predictions count as misses. The closed-set candidate limitation described above still applies.

In [7]:
subprocess.run([
    sys.executable, str(root / 'score.py'),
    '--gold', str(gold_path), '--pred', str(predictions_path),
    '--mms', str(mms_path), '--per-note', str(per_note_path),
], cwd=root, check=True);

notes 10  missing predictions 0  invalid codes 0  duplicate ids 0
slice A: n=10  top1 0.000  top3 0.000  points 0.0 / 75
SCORE 0.0 / 75


## Inspect per-note results and similarity scores

In [8]:
with per_note_path.open(encoding='utf-8') as file:
    results = [json.loads(line) for line in file if line.strip()]
with similarity_path.open(encoding='utf-8') as file:
    details = {row['id']: row for row in (json.loads(line) for line in file if line.strip())}
columns_to_show = ('id', 'gold', 'pred', 'top1', 'top3', 'ranked', 'matched_conditions', 'unmatched_conditions')
table_header = ''.join(f'<th>{html.escape(column)}</th>' for column in columns_to_show)
table_rows = []
for result in results:
    row = {**result, **details.get(result['id'], {})}
    cells = ''.join(f'<td>{html.escape(str(row.get(column, "")))}</td>' for column in columns_to_show)
    table_rows.append(f'<tr>{cells}</tr>')
print(f'{len(results)} notes evaluated')
display(HTML(f'<table><thead><tr>{table_header}</tr></thead><tbody>{"".join(table_rows)}</tbody></table>'))

10 notes evaluated


id,gold,pred,top1,top3,ranked,matched_conditions,unmatched_conditions
dev-051,['5B52'],"['1F40.Z', '1A07.Z', 'JA24.1']",False,False,"[{'code': '1F40.Z', 'similarity': 0.25}, {'code': '1A07.Z', 'similarity': 0.23570226039551587}, {'code': 'JA24.1', 'similarity': 0.23570226039551587}]","['pedal edema', 'poor appetite']","['severe acute malnutrition', 'recurrent diarrhea', 'sparse hair', 'flaky skin']"
dev-052,['BD10'],"['BA00.0', '1C62.0', 'BC81.3Z']",False,False,"[{'code': 'BA00.0', 'similarity': 0.7071067811865475}, {'code': '1C62.0', 'similarity': 0.5}, {'code': 'BC81.3Z', 'similarity': 0.4082482904638631}]","['hypertension', 'shortness of breath']","['congestive heart failure exacerbation', 'orthopnea', 'paroxysmal nocturnal dyspnea', 'leg swelling', 'pulmonary crackles', 'jugular venous distension', 'hepatomegaly', 'weight gain']"
dev-053,['QA42.0'],"['DA42.Z', 'JB64.0']",False,False,"[{'code': 'DA42.Z', 'similarity': 0.35355339059327373}, {'code': 'JB64.0', 'similarity': 0.2886751345948129}]","['nausea', 'pregnancy']",[]
dev-054,['5A40.0'],"['5A11', '1A40.Z']",False,False,"[{'code': '5A11', 'similarity': 0.4472135954999579}, {'code': '1A40.Z', 'similarity': 0.4082482904638631}]",['thirst'],"['prediabetes', 'elevated fasting glucose']"
dev-055,['BA40.1'],"['CA23.30', '5A11', 'FB53.0']",False,False,"[{'code': 'CA23.30', 'similarity': 0.35355339059327373}, {'code': '5A11', 'similarity': 0.31622776601683794}, {'code': 'FB53.0', 'similarity': 0.31622776601683794}]","['chest tightness', 'type 2 diabetes']",['stable angina']
dev-056,['JA63.2'],[],False,False,[],[],['gestational diabetes']
dev-057,['5A00.2Z'],"['DB60.Z', 'DD91.1', 'DA09.60']",False,False,"[{'code': 'DB60.Z', 'similarity': 0.2886751345948129}, {'code': 'DD91.1', 'similarity': 0.2581988897471611}, {'code': 'DA09.60', 'similarity': 0.23570226039551587}]","['constipation', 'facial swelling', 'fatigue']","['primary hypothyroidism', 'cold intolerance', 'weight gain', 'heavy menstrual bleeding', 'dry skin', 'bradycardia', 'delayed ankle reflex relaxation']"
dev-058,['BD90.0'],"['CA00', 'CA07.0', '1E90.0']",False,False,"[{'code': 'CA00', 'similarity': 0.5773502691896258}, {'code': 'CA07.0', 'similarity': 0.5773502691896258}, {'code': '1E90.0', 'similarity': 0.4082482904638631}]","['fever', 'throat redness']","['cervical lymphadenitis', 'right neck swelling']"
dev-059,['JA62.1'],"['GB51', 'GC08.Z', 'GC08.0']",False,False,"[{'code': 'GB51', 'similarity': 0.565685424949238}, {'code': 'GC08.Z', 'similarity': 0.4}, {'code': 'GC08.0', 'similarity': 0.22360679774997896}]","['dysuria', 'hematuria', 'pregnancy', 'pyuria', 'urinary frequency']","['cystitis', 'suprapubic discomfort']"
dev-060,['5B81.01'],['DA22.Z'],False,False,"[{'code': 'DA22.Z', 'similarity': 0.4082482904638631}]",['obesity'],['weight gain']
